In [1]:
import h5py
import numpy as np

targets = np.array([np.repeat(i, 3000) for i in range(10)]).flatten()

# Generated array is of dimensions
# (10, 60, 50, 48000)
# However, we are gonna collapse this into 2 dimensions only,
# turning it into a (30000, 48000)
inputs = np.zeros((30000, 48000), np.int16)
with h5py.File(".data/data.hdf5", "r") as f:
    for i, digit_group in enumerate(f.keys()):
        for j, speaker_group in enumerate(f[digit_group].keys()):
            for k, audio in enumerate(f[digit_group][speaker_group]):
                inputs[i * 3000 + j * 50 + k, :] = f[digit_group][speaker_group][audio][:]

In [7]:
import sklearn
import sklearn.decomposition
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from tqdm import tqdm

np.random.seed(42)
idx_permutation = np.random.permutation(30000)
inputs_permutated = inputs[idx_permutation, :]
targets_permutated = targets[idx_permutation]
SAMPLE_FRACTION = 1 / 3
SAMPLE_SIZE = int(SAMPLE_FRACTION * 30000)

train_inputs, test_inputs, train_targets, test_targets = train_test_split(
    inputs_permutated[:SAMPLE_SIZE], targets_permutated[:SAMPLE_SIZE], test_size=0.2
)

dimensions = np.array([2, 5, 10, 25, 50, 100, 250, 500, 1000, 2500, 5000])
stats = np.zeros((len(dimensions), 4))
for i, dimension in tqdm(enumerate(dimensions)):
    pca = sklearn.decomposition.PCA(n_components=dimension)
    pca.fit(train_inputs)
    error_train = np.mean(np.abs(train_inputs - pca.inverse_transform(pca.transform(train_inputs))))
    error_test = np.mean(np.abs(test_inputs - pca.inverse_transform(pca.transform(test_inputs))))
    stats[i, 0] = error_train
    stats[i, 1] = error_test
    classifier = SVC()
    classifier.fit(pca.transform(train_inputs), train_targets)
    stats[i, 2] = classifier.score(pca.transform(train_inputs), train_targets)
    stats[i, 3] = classifier.score(pca.transform(test_inputs), test_targets)
    print("Ended iteration for dimension", dimension)
    del pca

1it [00:13, 13.85s/it]

Ended iteration for dimension 2


2it [00:28, 14.44s/it]

Ended iteration for dimension 5


3it [00:43, 14.64s/it]

...

Ended iteration for dimension 1000


10it [12:23, 156.62s/it]

Ended iteration for dimension 2500


11it [23:59, 130.85s/it]

Ended iteration for dimension 5000


In [8]:
import pandas as pd

result_idx = pd.Index(dimensions, name="Dimensions")
result_columns = pd.Index(["MAE-Train", "MAE-Test", "Accuracy-Train", "Accuracy-Test"], name="Error")
result = pd.DataFrame(stats, index=result_idx, columns=result_columns)
display(result)

Error,MAE-Train,MAE-Test,Accuracy-Train,Accuracy-Test
Dimensions,,,,
2,53.630551,54.742387,0.128125,0.1095
5,53.699957,54.929059,0.150250,0.1160
10,53.718825,55.129428,0.157875,0.1240
25,53.197360,55.168070,0.192500,0.1395
50,51.509735,54.323844,0.223500,0.1450
100,47.865147,52.187569,0.278750,0.1570
250,36.484853,42.221782,0.405625,0.1890
500,23.351953,28.819457,0.493625,0.1985
1000,15.156788,21.023191,0.519250,0.1950
